## Task


The aim of the assignment is to apply the NLP techniques you have learnt in class to analyse one of the 
datasets described below.   
• Note: some of the datasets are quite large, so you may need to sample a small percentage of the 
data and work that. 
The exact tasks performed may depend on the dataset chosen, but we would expect to see some of the 
following: 
#### 1. Preliminary analysis:  
Briefly describe the data:

What is the structure of the dataset? What type of task was the dataset collected for? 

What type of documents does it contain? How many are there? How long are they on average and 

what is their distribution? 

How big is the vocabulary of the collection? How big is the vocabulary of a document on average? 

Play around with documents using code from the early parts of the course. For example, you could:
Cluster the documents, visualise the clusters and to try to understand what types of groups are 
present.

Index the documents so that you can perform keyword search over them. 

Train a Word2Vec embedding and investigate the properties of the resulting embedding. 
#### 2. Training models: 
Each dataset has been created with a particular task in mind. You don’t necessarily need to tackle that 
particular problem, but you do need to train some model(s) on the data:

train ML models (e.g. a linear classifier, an LSTM and/or a Transformer) to perform a particular 

task on the data; 
if possible, try to fine-tune a pretrained models on the same task and compare their performance; 

try an LLM on the task, comparing one, few and zero-shot performance;  
and perhaps even try to fine-tune a small LLM on the task (if it makes sense to do so).   
#### 3. Possible extensions: 
Depending on the dataset chosen there will be many additional investigations that you could perform, 
for example:  
- investigate another task on the same dataset  
- investigate the same task on a related dataset 
- use text-to-speech and speech-to-text models to create a voice interactive chatbot
- create your own dialog dataset by transcribing audio conversations (e.g. using MS Teams).  

# 1. Preliminary analysis

## 1.a) Data Exploration

### Import the dataset and see how data are stored inside of it

Import the general necessary libraries ( the specific libraries will be imported later in the cells where they are used)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import random
from collections import Counter
from textblob import TextBlob
from tqdm import tqdm

In [ ]:
from datasets import load_dataset

dataset = load_dataset("neural-bridge/rag-dataset-12000")

In [ ]:
dataset

They are already divided into train and test set. So we just need to convert them to Pandas Dataframes.

In [ ]:
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

In [ ]:
print('Train dataset len:', len(train_df))
print('Test dataset len:', len(test_df))

Let's then print a complete (question, context, answer) sample of our dataset

In [ ]:
# Show an example

context = train_df['context'][1]
question = train_df['question'][1]
answer = train_df['answer'][1]

print(f"\nQuestion\n")
print(f"{question}\n")
print("------------------------------------------------------------------------")
#print(f"\nContext\n")
#print(f"{context}\n")
print("------------------------------------------------------------------------")
print(f"\nAnswer\n\n")
print(answer)

In [ ]:
# Count NaN / null values per column for train set
train_null_counts = train_df.isna().sum()

# Count NaN / null values per column for test set
test_null_counts = test_df.isna().sum()

null_counts = train_null_counts + test_null_counts
# Print the result
print(f"NaN/null values per column: \n{null_counts}")
print(f"NaN/null values per column in train: \n{train_null_counts}")
print(f"NaN/null values per column in test: \n{test_null_counts}")


train_df = train_df.dropna()
test_df = test_df.dropna()

print('Any missing values: ')
print(train_df.isnull().any() & test_df.isnull().any())

### Tokenize

Let's tokenize our text before running analysis on it

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer

nltk.download('punkt')
nltk.download('stopwords')



stop_words = set(stopwords.words('english'))
# stemmer = PorterStemmer()

context_lengths = []
question_lengths = []
vocab = set()

all_tokens = []

train_df_tokenized = []

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalnum()]  # Remove punctuation
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    # tokens = [stemmer.stem(word) for word in tokens]  # Apply stemming
    return tokens

def append_tokens(entry):
    context_tokens = preprocess(entry['context'])
    question_tokens = preprocess(entry['question'])
    answer_tokens = preprocess(entry['answer'])
    
    context_lengths.append(len(context_tokens))
    question_lengths.append(len(question_tokens))
    vocab.update(context_tokens)
    vocab.update(question_tokens)

    all_tokens.extend(context_tokens)
    all_tokens.extend(question_tokens)

    train_df_tokenized.append({'context': context_tokens, 'question': question_tokens, 'answer': answer_tokens})


train_df.apply(lambda entry: append_tokens(entry), axis=1)

In [ ]:
train_df_tokenized = pd.DataFrame(train_df_tokenized, columns=['context', 'question', 'answer'])
train_df_tokenized.head()

### Analysis of Token Frequencies (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_df_tokenized['context_str'] = train_df_tokenized['context'].apply(lambda tokens: ' '.join(tokens))
train_df_tokenized['question_str'] = train_df_tokenized['question'].apply(lambda tokens: ' '.join(tokens))

# TF-IDF on contexts only
vectorizer_context = TfidfVectorizer()
tfidf_context = vectorizer_context.fit_transform(train_df_tokenized['context_str'])

# TF-IDF on questions only
vectorizer_question = TfidfVectorizer()
tfidf_question = vectorizer_question.fit_transform(train_df_tokenized['question_str'])

# TF-IDF on combined contexts + questions
train_df_tokenized['combined'] = train_df_tokenized['context_str'] + ' ' + train_df_tokenized['question_str']
vectorizer_combined = TfidfVectorizer()
tfidf_combined = vectorizer_combined.fit_transform(train_df_tokenized['combined'])

We can see that the tf-idf matrix is much thinner than the question one. This is natural since the context has a much bigger vocabulary. 
\
Another interesting thing to point out is that the matrix due to the combination of both question and context is slightly wider than the context one because most of the words contained in the questions are also contained in the context.

In [ ]:
tfidf_question_df = pd.DataFrame(tfidf_question.toarray(), columns=vectorizer_question.get_feature_names_out())
tfidf_question_df

In [ ]:
tfidf_context_df = pd.DataFrame(tfidf_context.toarray(), columns=vectorizer_context.get_feature_names_out())
tfidf_context_df

In [ ]:
tfidf_combined_df = pd.DataFrame(tfidf_combined.toarray(), columns=vectorizer_combined.get_feature_names_out())
tfidf_combined_df

Let's visualize through a histogram the most frequent word in the contexts

In [ ]:
word_scores = tfidf_context_df.sum(axis=0).sort_values(ascending=False)

top_n = 20
top_words = word_scores.head(top_n)

plt.figure(figsize=(10, 6))
top_words.plot(kind='bar')
plt.title(f"Top {top_n} Words by TF-IDF Score")
plt.xlabel("Words")
plt.ylabel("Total TF-IDF Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
feature_names = vectorizer_context.get_feature_names_out()

def get_top_n_words(row, n):
    row_data = row.toarray().flatten()
    top_indices = row_data.argsort()[::-1][:n]
    
    return [(feature_names[i], row_data[i]) for i in top_indices if row_data[i] > 0]

N = 5
train_df_tokenized[f'top_{N}_tfidf_words_context'] = [
    get_top_n_words(tfidf_context[i], N) for i in range(tfidf_context.shape[0])
]

In [ ]:
train_df_tokenized.head()

Let's see the most frequent words (calculated through  tf-idf score) for each context

In [ ]:
def plot_top_words_with_scores(word_score_pairs, title='Top TF-IDF Words'):
    words, scores = zip(*word_score_pairs)
    plt.figure(figsize=(5, 4))
    plt.barh(words[::-1], scores[::-1], color='red')  # reverse for descending order
    plt.xlabel('TF-IDF Score')
    plt.title(title)
    plt.tight_layout()
    plt.show()

for i in range(10):
    plot_top_words_with_scores(train_df_tokenized.loc[i, f'top_{N}_tfidf_words_context'],
                           title=f'Document {i+1} - Top TF-IDF Words')

Now instead let's visualize the most frequent tokens contained in the train set including all of the columns (question, context, answer)

In [ ]:
fdist = FreqDist(all_tokens)
fdist

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400).generate_from_frequencies(fdist)
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Vocabulary")
plt.show()

In [ ]:
fdist.plot(30,  title="Top 30 Most Frequent Words")

There are lots of words that appear just once. Let's count them

In [ ]:
least_freq_1_words = [word for word, freq in fdist.items() if freq == 1 and word.isalpha()]
print(len(least_freq_1_words))

These are some of the words that appear only once 

Now we'd like to produce some statistics on the length of the contexts and questions

In [ ]:
print(f"Number of documents: {len(train_df)}")
print(f"Average context length: {sum(context_lengths)/len(context_lengths):.2f} tokens")
print(f"Average question length: {sum(question_lengths)/len(question_lengths):.2f} tokens")
print(f"Vocabulary size: {len(vocab)}")

sns.histplot(context_lengths, bins=50, kde=True)
plt.title('Distribution of Context Lengths')
plt.xlabel('Number of Tokens')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sns.histplot(question_lengths, bins=50, kde=True)
plt.title("Question Length Distribution")
plt.xlabel("Number of Tokens")
plt.ylabel("Frequency")
plt.show()

## 1.b) Data cleaning

* Stemming? necessary or not

In [ ]:
tokenized_contexts = train_df_tokenized['context']
tokenized_questions = train_df_tokenized['question']
tokenized_answers = train_df_tokenized['answer']

print(len(tokenized_questions))
print(len(tokenized_contexts))
print(len(tokenized_answers))

# Optionally filter out short sentences (more than 3 tokens)
#tokenized_questions = tokenized_questions[tokenized_questions.apply(lambda x: len(x) > 3)]
#tokenized_contexts = tokenized_contexts[tokenized_contexts.apply(lambda x: len(x) > 3)]
#tokenized_answers = tokenized_answers[tokenized_answers.apply(lambda x: len(x) > 3)]

print(len(tokenized_questions))
print(len(tokenized_contexts))
print(len(tokenized_answers))

Create the vector to feed the Word2Vec model in such a way to have (question, context, answer). \
There are other possibilities to form the vector to feed to the model (e. g. the one commented out ) they all perform pretty well and more or less the same

In [ ]:
all_tokenized_sentences = [ c + q + a for c, q, a in zip(tokenized_contexts.tolist(), tokenized_questions.tolist(), tokenized_answers.tolist()) ]

# all_tokenized_sentences = tokenized_contexts.tolist() + tokenized_questions.tolist() + tokenized_answers.tolist() # ther possibility performing worse

### Drop the words occurring once

Convert least_freq_1_words to a set because it is much faster to search in it and then filter out all the words that occurr once (N.T. all the words not the numbers)

In [ ]:
least_freq_1_words = set(least_freq_1_words)
all_tokenized_sentences = [ [ word for word in sentence if word not in least_freq_1_words ] for sentence in all_tokenized_sentences ]


### Create embeddings with Word2Vec 



In [ ]:
from gensim.models.word2vec import Word2Vec

d = 30 # dimension of the embeddings

model = Word2Vec(all_tokenized_sentences, vector_size=d, min_count=5, window=10) # default model is skipgram (it tends to perform better)

In [ ]:
len(model.wv)

# print(model.wv.key_to_index) # output is too long

In [ ]:
term = 'house'
model.wv[term]
model.wv.most_similar(term)

In [ ]:
seed = random.seed(0)

sample = random.sample(list(model.wv.key_to_index), 1000)
word_vectors = model.wv[sample]
word_vectors

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=3, n_iter=2000)
tsne_embedding = tsne.fit_transform(word_vectors)

In [ ]:
x, y, z = np.transpose(tsne_embedding)

In [ ]:
fig = px.scatter_3d(x=x, y=y, z=z)
fig.update_traces(marker=dict(size=3,line=dict(width=2)))
fig.show()

In [ ]:
fig = px.scatter_3d(x=x[:200],y=y[:200],z=z[:200],text=sample[:200])
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

In [ ]:
colours = ['red','green','blue','orange','yellow','purple','pink','cream','brown','black','white','gray']

word_vectors = model.wv[colours+sample]

tsne = TSNE(n_components=3)
tsne_embedding = tsne.fit_transform(word_vectors)

x, y, z = np.transpose(tsne_embedding)

In [ ]:
r = (-200,200)
fig = px.scatter_3d(x=x, y=y, z=z, range_x=r, range_y=r, range_z=r, text=colours + [None] * 1000)
fig.update_traces(marker=dict(size=3,line=dict(width=2)),textfont_size=10)
fig.show()

### 1.c) Clustering

Now that we have the embeddings of the words, we would like to make operations fast with them. \
Faiss is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning. \
Faiss is written in C++ with complete wrappers for Python/numpy. Some of the most useful algorithms are implemented on the GPU. It is developed primarily at Meta's Fundamental AI Research group. \
Let's install it and try to make a search!

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
index = faiss.IndexFlatL2(d) 

In [ ]:
# This creates a matrix with the embeddings of the words as rows
words = list(model.wv.key_to_index.keys())
word_vectors = np.array([model.wv[word] for word in words]).astype("float32") 

# This creates an index 
index = faiss.IndexFlatL2(word_vectors.shape[1])
index.add(word_vectors)

In [ ]:
# Let's perform a search
query_word = 'house'
query_vector = np.array([model.wv[query_word]]).astype("float32")
D, I = index.search(query_vector, k=10)  # D = distances, I = indices
similar_words = [words[i] for i in I[0]]
print(f"Words similar to '{query_word}':", similar_words)

Let's create a function that shows examples of clusters

In [ ]:
def show_cluster_examples(column, n_clusters=5, top_n=30):
    if column == "question": 
        df = pd.DataFrame({column: tokenized_questions, 'cluster': labels})   
    elif column == "context": 
        df = pd.DataFrame({column: tokenized_context, 'cluster': labels})
    elif column == "answer": 
        df = pd.DataFrame({column: tokenized_answers, 'cluster': labels})
    
    unique_clusters = sorted(df['cluster'].unique()) # Get unique labels and sort them
    clusters_to_show = unique_clusters[:n_clusters]

    for i in clusters_to_show:
        print(f"\n Cluster {i} (size={len(df[df.cluster==i])}):")
        examples = df[df.cluster == i][column].sample(min(top_n, len(df[df.cluster == i])), random_state=0)
        for q in examples:
            print("  -", q)

Let's now use the embeddings of the words provided by Word2Vec. \
The Word2Vec gives us the embeddings of words, now we have to compute the embeddings of sentences to then compare them and cluster them. 
We use the mean of the embedding of the words to form the embeddings of the sentence (N.T. We could have also used the sum).

In [ ]:
def sentence_embedding(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

question_embeddings = np.array([sentence_embedding(tokens, model) for tokens in tokenized_questions])
context_embeddings = np.array([sentence_embedding(tokens, model) for tokens in tokenized_contexts])
answer_embeddings = np.array([sentence_embedding(tokens, model) for tokens in tokenized_answers])

This is a function to guess the magnitude of k. It works in this way, we calculate the cosine similarity of the sentences embeddings and count them 

In [ ]:
from sklearn.cluster import KMeans

def cosine_similarity_faiss(vectors):
    vectors = np.asarray(vectors).astype('float32')
    normalized = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    similarity_matrix = np.matmul(normalized, normalized.T) 
    return similarity_matrix

def extract_similar_pairs(sim_matrix, threshold):
    n = sim_matrix.shape[0]
    similar_pairs = []

    for i in range(1, n):
        for j in range(i):
            if sim_matrix[i, j] > threshold:
                similar_pairs.append((i, j, sim_matrix[i, j]))
    l = len(similar_pairs)
    print(f"Sim pairs: {l}")
    return l

def sim_matrix_density(embeddings, threshold=0.8):
    S = cosine_similarity_faiss(embeddings)
    print(S.shape)
    l = extract_similar_pairs(S, threshold)
    return l/(S.shape[0] * S.shape[1])*100, S

The idea of having the sim_matrix_density function is to have an idea of which percentage of the sentences is similar. \
In this way it is possible to get a small indicator of the number of the topics. \
The bigger the percentage the less the topics

In [ ]:
perc, S = sim_matrix_density(question_embeddings, 0.7) 

In [ ]:
print(f"The percentage of similar entries of the similarity matrix is {perc:.2f}%")

In [ ]:
S = cosine_similarity_faiss(question_embeddings)

upper_triangle = S[np.triu_indices_from(S)]

plt.hist(upper_triangle, bins=50, edgecolor='black')
plt.title('Distribution of Cosine Similarity Scores')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()

### Elbow method

In [ ]:
k_values = range(80, 500, 10)
wcss = []

for k in tqdm(k_values):
    kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
    kmeans.fit(question_embeddings)
    wcss.append(kmeans.inertia_)

plt.plot(k_values, wcss, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method on Sentence Embeddings")
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.xticks(range(min(k_values), max(k_values) + 1, 10))
plt.show()

In [ ]:
k = 250

In [ ]:
# 250 - 300 working fine
kmeans = KMeans(n_clusters=k, random_state=0)
labels = kmeans.fit_predict(question_embeddings)

In [ ]:
show_cluster_examples('question')

In [ ]:
kmeans = KMeans(n_clusters=k, random_state=0)
labels = kmeans.fit_predict(answer_embeddings)

In [ ]:
show_cluster_examples('answer')

### Let's now use a Transformer to create the embeddings 

In [ ]:
print(tokenized_questions.tolist()[-10:])

In [ ]:
tokenized_questions = tokenized_questions[tokenized_questions.apply(lambda x: len(x) > 3)]
tokenized_contexts = tokenized_contexts[tokenized_contexts.apply(lambda x: len(x) > 3)]
tokenized_answers = tokenized_answers[tokenized_answers.apply(lambda x: len(x) > 3)]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2") 
question_embeddings = model.encode(tokenized_questions.tolist(), show_progress_bar=True)

In [ ]:
#k, S = guess_k_for_clustering(question_embeddings, 0.99)
print(k)

In [ ]:
upper_triangle = S[np.triu_indices_from(S)]

plt.hist(upper_triangle, bins=50, edgecolor='black')
plt.title('Distribution of Cosine Similarity Scores')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=300, random_state=0)
labels = kmeans.fit_predict(question_embeddings)

In [ ]:
show_cluster_examples('question', 10)

In [ ]:
show_cluster_examples('answer')

In [ ]:
#show_cluster_examples('context')

In [ ]:
answer_embeddings = model.encode(tokenized_answers.tolist(), show_progress_bar=True)


In [ ]:
kmeans = KMeans(n_clusters=300, random_state=0)
labels = kmeans.fit_predict(answer_embeddings)

In [ ]:
show_cluster_examples('answer')

# Training models

## Claudia
### Sottotitolo 1
Inserisci qui il tuo testo

## Edoardo

## Alberto

## Martina

# Extensions

## Chatbot

# Conclusions